### Retrieve Author Country Information Using OpenAlex

In [ ]:
import numpy as np
import pandas as pd
import yaml

In [ ]:
import os
os.chdir('../../')

In [ ]:
from urllib.parse import urlparse

def extract_country_tld(url):
    """
    Extracts the country code TLD from a URL if present.
    For example, returns '.jp' for a URL ending in '.jp'.
    """
    try:
        hostname = urlparse(url).hostname
        if hostname:
            parts = hostname.split('.')
            # If the last part has exactly 2 alphabetic characters, assume it's a country TLD.
            if parts[-1].isalpha() and len(parts[-1]) == 2:
                return parts[-1].upper()
        return None
    except Exception as e:
        return None

In [ ]:
# Load configuration settings from a YAML file.
# The configuration file typically contains file paths and other parameters.
dataset_config = yaml.safe_load(open("config/dataset.yml"))
dataset_config

In [ ]:
institutions_oa = pd.read_csv(dataset_config['path_other'] + 'OpenAlex_institutions.csv').rename(columns={'id': 'institution_id', 'country_code': 'ctry'})
institutions_oa

In [ ]:
institutions_oa['institution_id'] = institutions_oa['institution_id'].str.replace(r'^I', '', regex=True).astype(pd.Int64Dtype())
institutions_oa

In [ ]:
country_abbr_db = pd.read_csv(dataset_config['path_other'] + 'ctry_country.csv')
country_abbr_db

## Recognize country from OpenAlex matched results

(#01)   All data []
(#02)   ├─ With ctry code []
(#03)   │    ├─ With valid country []          --> dfs_with_ctry
(#04)   │    └─ Without valid country []
(#05)   ├─ With inst ID []
(#06)   │    ├─ With inst-ctry []              --> dfs_with_ctry
(#07)   │    └─ Without ctry []
(#08)   │         ├─ With url suffix []
(#09)   │         │    ├─ Valid url ctry []    --> dfs_with_ctry
(#10)   │         │    └─ Without url ctry []  --> dfs_with_aff
(#11)   │         └─ Without url suffix []     --> dfs_with_aff
(#12)   └─ With aff name []                    --> dfs_with_aff

In [ ]:
dfs_with_ctry = []
dfs_with_aff = []

In [ ]:
# 01 All data
relevant_data = pd.read_parquet(dataset_config['path_processed'] + 'CN_CN/OA2_relevant_data.parquet')
relevant_data

In [ ]:
# 02 With ctry code
relevant_with_ctry = relevant_data[~relevant_data.ctry.isna()].drop(columns=['institution_id', 'affiliation'])
relevant_with_ctry = relevant_with_ctry.merge(country_abbr_db, on='ctry', how='left')
relevant_with_ctry

In [ ]:
# 04 Without valid country
len(relevant_with_ctry[relevant_with_ctry.country.isna()])

In [ ]:
relevant_with_ctry[relevant_with_ctry.country.isna()].ctry.unique()

In [ ]:
# 03 With valid country
relevant_with_ctry_valid = relevant_with_ctry[~relevant_with_ctry.country.isna()].drop(columns=['ctry'])
relevant_with_ctry_valid

In [ ]:
# 03 --> dfs_with_ctry
dfs_with_ctry.append(relevant_with_ctry_valid)

In [ ]:
# 05 With inst ID
relevant_with_inst = relevant_data[~relevant_data.institution_id.isna()].drop(columns=['ctry', 'affiliation'])
relevant_with_inst

In [ ]:
# 12 With aff name
relevant_with_aff = relevant_data[~relevant_data.affiliation.isna()].drop(columns=['ctry', 'institution_id']).rename(columns={'affiliation': 'affiliationame'})
relevant_with_aff

In [ ]:
# 12 --> dfs_with_aff
dfs_with_aff.append(relevant_with_aff)

## Section A
### Generate `institution_id` and `country` matching

In [ ]:
instid_ctry_db = pd.merge(institutions_oa, country_abbr_db, on='ctry')
instid_ctry_db = instid_ctry_db.drop(columns='ctry')
instid_ctry_db

In [ ]:
rel_inst_ctry = relevant_with_inst.merge(instid_ctry_db, on='institution_id', how='left')
rel_inst_ctry

In [ ]:
# 06 With inst-ctry
rel_inst_with_ctry = rel_inst_ctry[~rel_inst_ctry.country.isna()].drop(columns=['institution_id'])
rel_inst_with_ctry

In [ ]:
# 06 --> dfs_with_ctry
dfs_with_ctry.append(rel_inst_with_ctry)

In [ ]:
# 07 Without ctry
rel_inst_wo_ctry = rel_inst_ctry[rel_inst_ctry.country.isna()].drop(columns=['country'])
rel_inst_wo_ctry

In [ ]:
instid_name_db = pd.read_csv(dataset_config['path_openalex'] + 'institutions.csv.gz', compression='gzip', usecols=['id', 'display_name', 'homepage_url']).rename(columns={'id': 'institution_id', 'display_name': 'affiliationame'})
instid_name_db

In [ ]:
instid_name_db['institution_id'] = instid_name_db['institution_id'].str.replace(r'^https://openalex.org/I', '', regex=True).astype(pd.Int64Dtype())
instid_name_db['url_suffix'] = instid_name_db['homepage_url'].apply(extract_country_tld)
instid_name_db

In [ ]:
instid_url_db_ctry = instid_name_db[['institution_id', 'url_suffix']].rename(columns={'url_suffix': 'ctry'}).dropna()
instid_url_db_ctry

In [ ]:
rel_inst_url = rel_inst_wo_ctry.merge(instid_url_db_ctry, on='institution_id', how='left')
rel_inst_url

In [ ]:
# 08 With url suffix
rel_url_matched = rel_inst_url[~rel_inst_url.ctry.isna()]
rel_url_result = rel_url_matched.merge(country_abbr_db, on='ctry', how='left')
rel_url_result

In [ ]:
rel_url_result[rel_url_result.country.isna()].ctry.unique()

In [ ]:
# 09 Valid url ctry
rel_url_result_with_ctry = rel_url_result[~rel_url_result.country.isna()]
rel_url_result_with_ctry

In [ ]:
# 09 --> dfs_with_ctry
dfs_with_ctry.append(rel_url_result_with_ctry.drop(columns=['institution_id', 'ctry']))

In [ ]:
# 10 Without url ctry
rel_url_result_without_ctry = rel_url_result[rel_url_result.country.isna()]
rel_url_result_without_ctry

In [ ]:
rel_url_without_ctry_aff = rel_url_result_without_ctry.merge(instid_name_db[['institution_id', 'affiliationame']], on='institution_id').drop(columns=['institution_id'])
rel_url_without_ctry_aff

In [ ]:
# 10 --> dfs_with_aff
dfs_with_aff.append(rel_url_without_ctry_aff.drop(columns=['ctry', 'country']))

In [ ]:
# 11 Without url suffix
rel_url_unmatched = rel_inst_url[rel_inst_url.ctry.isna()].drop(columns=['ctry'])
rel_url_unmatched

In [ ]:
rel_with_aff = rel_url_unmatched.merge(instid_name_db[['institution_id', 'affiliationame']], on='institution_id', how='left').drop(columns=['institution_id'])
rel_with_aff

In [ ]:
sum(rel_with_aff.affiliationame.isna())

In [ ]:
# 11 --> dfs_with_aff
dfs_with_aff.append(rel_with_aff)

In [ ]:
data_all_with_ctry = pd.concat(dfs_with_ctry, ignore_index=True).rename(columns={'author_id': 'authorid'})
data_all_with_ctry

In [ ]:
sum(data_all_with_ctry.country.isna())

In [ ]:
data_all_with_aff = pd.concat(dfs_with_aff, ignore_index=True).rename(columns={'author_id': 'authorid'})
data_all_with_aff

In [ ]:
sum(data_all_with_aff.affiliationame.isna())

## Export

In [ ]:
data_all_with_ctry.to_parquet(dataset_config['path_processed'] + 'CN_CN/OA3_recognized_by_openalex.parquet')

In [ ]:
data_all_with_aff.to_parquet(dataset_config['path_processed'] + 'CN_CN/OA3_missing_geo.parquet')

In [ ]:
f"OpenAlex country recognition (entries matched by OpenAlex): {len(data_all_with_ctry)} / {len(relevant_data)} = {len(data_all_with_ctry) / len(relevant_data) * 100:.4f}%"

In [ ]:
f"OpenAlex country recognition (all matches OA): {len(data_all_with_ctry)} / {len(data_all_with_ctry)+len(data_all_with_aff)} = {len(data_all_with_ctry) / (len(data_all_with_ctry)+len(data_all_with_aff)) * 100:.4f}%"